# 위치 임베딩 분석 & 보간 품질 평가
## 04_positional_embedding.ipynb

**목표**: ViT의 위치 임베딩 특성 분석 및 해상도 변경 시 보간 효과 검증

**실험 내용**:
- 학습된 절대 위치 임베딩 vs 2D sin-cos 임베딩 비교
- 2D 보간 전후 성능 변화 분석  
- 위치 임베딩 패턴 시각화

**산출물**:
- `pos_interp_curve_v2.png` - 해상도별 성능 변화 곡선
- `pe_heatmap_v2.png` - 위치 임베딩 히트맵 시각화
- `pos_embedding_analysis_v2.csv` - 보간 품질 정량 분석

**소요시간**: ~10분


In [ ]:
# 필수 라이브러리 및 설정
import os
import sys
import warnings
warnings.filterwarnings('ignore')

sys.path.append('..')
os.makedirs('../reports/figures', exist_ok=True)
os.makedirs('../reports/tables', exist_ok=True)

import torch
import torch.nn.functional as F
import timm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
from typing import Tuple, Dict, List
from tqdm import tqdm

# 시각화 설정
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("viridis")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

# 랜덤 시드 고정
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 디바이스: {device}")

# 실험 설정
TEST_RESOLUTIONS = [
    (224, 224),   # 원본
    (288, 288),   # 1.28x
    (320, 224),   # 직사각형 1
    (384, 384),   # 1.71x  
    (448, 320),   # 직사각형 2
    (512, 512),   # 2.29x
]

print(f"📐 테스트 해상도: {len(TEST_RESOLUTIONS)}개")
for h, w in TEST_RESOLUTIONS:
    patches = (h // 16) * (w // 16)
    print(f"   {h}×{w} → {patches} 패치")


In [ ]:
# 위치 임베딩 분석 클래스
class PositionalEmbeddingAnalyzer:
    """위치 임베딩 분석 및 비교 클래스"""
    
    def __init__(self, model_name: str = 'vit_base_patch16_224.augreg_in21k_ft_in1k'):
        self.model = timm.create_model(model_name, pretrained=True)
        self.model = self.model.to(device)
        self.model.eval()
        
        # 원본 정보 저장
        self.original_pos_embed = self.model.pos_embed.clone()
        self.patch_size = self.model.patch_embed.patch_size[0]
        self.embed_dim = self.model.embed_dim
        self.original_grid_size = int(np.sqrt(self.original_pos_embed.shape[1] - 1))
        
        print(f"✅ 모델 로드: {model_name}")
        print(f"📊 원본 위치 임베딩: {self.original_pos_embed.shape}")
        print(f"🎯 패치 크기: {self.patch_size}")
        print(f"📐 원본 그리드: {self.original_grid_size}×{self.original_grid_size}")
    
    def create_2d_sincos_embedding(self, grid_h: int, grid_w: int) -> torch.Tensor:
        """2D sin-cos 위치 임베딩 생성"""
        embed_dim = self.embed_dim
        
        # 그리드 좌표 생성
        grid_h_coords = np.arange(grid_h, dtype=np.float32)
        grid_w_coords = np.arange(grid_w, dtype=np.float32)
        grid = np.meshgrid(grid_w_coords, grid_h_coords)  # [2, H, W]
        grid = np.stack(grid, axis=0)  # [2, H, W]
        
        # 주파수 생성
        omega = np.arange(embed_dim // 4, dtype=np.float32)
        omega = 1. / (10000 ** (omega / (embed_dim // 4)))
        
        # Sin-cos 임베딩 계산
        pos_embed = np.zeros((grid_h, grid_w, embed_dim))
        
        for i in range(2):  # x, y 좌표
            pos = grid[i]  # [H, W]
            for j, w_freq in enumerate(omega):
                pos_embed[:, :, i * (embed_dim // 4) + j * 2] = np.sin(pos * w_freq)
                pos_embed[:, :, i * (embed_dim // 4) + j * 2 + 1] = np.cos(pos * w_freq)
        
        # [1, H*W, embed_dim] 형태로 변환
        pos_embed = pos_embed.reshape(1, grid_h * grid_w, embed_dim)
        
        # CLS 토큰 임베딩 추가 (0으로 초기화)
        cls_embed = np.zeros((1, 1, embed_dim))
        pos_embed = np.concatenate([cls_embed, pos_embed], axis=1)
        
        return torch.from_numpy(pos_embed).float().to(device)
    
    def interpolate_pos_embedding(self, target_h: int, target_w: int, 
                                 method: str = 'bicubic') -> torch.Tensor:
        """위치 임베딩 보간"""
        patch_h = target_h // self.patch_size
        patch_w = target_w // self.patch_size
        
        if patch_h == self.original_grid_size and patch_w == self.original_grid_size:
            return self.original_pos_embed
        
        # CLS 토큰 분리
        cls_pos_embed = self.original_pos_embed[:, :1]
        patch_pos_embed = self.original_pos_embed[:, 1:].reshape(
            1, self.original_grid_size, self.original_grid_size, self.embed_dim
        )
        
        # 보간 수행
        patch_pos_embed = patch_pos_embed.permute(0, 3, 1, 2)  # [1, dim, H, W]
        patch_pos_embed = F.interpolate(
            patch_pos_embed, 
            size=(patch_h, patch_w), 
            mode=method, 
            align_corners=False
        )
        patch_pos_embed = patch_pos_embed.permute(0, 2, 3, 1).reshape(
            1, patch_h * patch_w, self.embed_dim
        )
        
        # CLS 토큰과 결합
        interpolated = torch.cat([cls_pos_embed, patch_pos_embed], dim=1)
        return interpolated
    
    def measure_interpolation_quality(self, target_h: int, target_w: int) -> Dict:
        """보간 품질 측정"""
        # 원본 임베딩
        original = self.original_pos_embed[:, 1:]  # CLS 제외
        
        # 보간된 임베딩
        interpolated = self.interpolate_pos_embedding(target_h, target_w)[:, 1:]
        
        # 다시 원본 크기로 보간하여 비교
        patch_h = target_h // self.patch_size
        patch_w = target_w // self.patch_size
        
        back_interpolated = interpolated.reshape(1, patch_h, patch_w, self.embed_dim)
        back_interpolated = back_interpolated.permute(0, 3, 1, 2)
        back_interpolated = F.interpolate(
            back_interpolated,
            size=(self.original_grid_size, self.original_grid_size),
            mode='bicubic',
            align_corners=False
        )
        back_interpolated = back_interpolated.permute(0, 2, 3, 1).reshape(
            1, self.original_grid_size * self.original_grid_size, self.embed_dim
        )
        
        # 품질 지표 계산
        mse = F.mse_loss(back_interpolated, original).item()
        cosine_sim = F.cosine_similarity(
            back_interpolated.flatten(), original.flatten(), dim=0
        ).item()
        l2_norm_diff = torch.norm(back_interpolated - original).item()
        
        return {
            'resolution': f"{target_h}x{target_w}",
            'target_patches': patch_h * patch_w,
            'mse_loss': mse,
            'cosine_similarity': cosine_sim,
            'l2_norm_diff': l2_norm_diff,
            'interpolation_ratio': (patch_h * patch_w) / (self.original_grid_size ** 2)
        }

# 분석기 초기화
analyzer = PositionalEmbeddingAnalyzer()
print("✅ 위치 임베딩 분석기 초기화 완료")


In [ ]:
# 보간 품질 분석 실행
print("🔬 보간 품질 분석 시작...")

interpolation_results = []

for h, w in tqdm(TEST_RESOLUTIONS, desc="해상도별 보간 품질 측정"):
    result = analyzer.measure_interpolation_quality(h, w)
    interpolation_results.append(result)
    
    print(f"   {h}×{w}: MSE={result['mse_loss']:.4f}, "
          f"Cosine={result['cosine_similarity']:.4f}")

# 결과를 DataFrame으로 변환
interp_df = pd.DataFrame(interpolation_results)

print(f"\n📊 보간 품질 분석 완료:")
print(f"   평균 MSE: {interp_df['mse_loss'].mean():.4f}")
print(f"   평균 Cosine 유사도: {interp_df['cosine_similarity'].mean():.4f}")
print(f"   최대 L2 차이: {interp_df['l2_norm_diff'].max():.4f}")

# 결과 저장
interp_df.to_csv('../reports/tables/pos_embedding_analysis_v2.csv', index=False)
print("💾 결과 저장: pos_embedding_analysis_v2.csv")


In [ ]:
# 위치 임베딩 시각화
print("🎨 위치 임베딩 시각화 생성 중...")

fig, axes = plt.subplots(3, 2, figsize=(16, 20))

# 1. 원본 위치 임베딩 히트맵
ax1 = axes[0, 0]
original_patches = analyzer.original_pos_embed[0, 1:].cpu().numpy()  # CLS 제외
original_2d = original_patches.reshape(analyzer.original_grid_size, 
                                      analyzer.original_grid_size, 
                                      analyzer.embed_dim)

# 차원별 평균으로 단순화
heatmap_data = np.mean(original_2d, axis=2)
im1 = ax1.imshow(heatmap_data, cmap='viridis', interpolation='bilinear')
ax1.set_title('원본 위치 임베딩 (14×14)', fontsize=14, fontweight='bold')
ax1.set_xlabel('패치 X 좌표')
ax1.set_ylabel('패치 Y 좌표')
plt.colorbar(im1, ax=ax1, fraction=0.046)

# 2. 보간된 위치 임베딩 (448×320)
ax2 = axes[0, 1]
interp_embed = analyzer.interpolate_pos_embedding(448, 320)
interp_patches = interp_embed[0, 1:].cpu().numpy()
patch_h, patch_w = 448 // 16, 320 // 16
interp_2d = interp_patches.reshape(patch_h, patch_w, analyzer.embed_dim)
interp_heatmap = np.mean(interp_2d, axis=2)

im2 = ax2.imshow(interp_heatmap, cmap='viridis', interpolation='bilinear')
ax2.set_title(f'보간된 위치 임베딩 ({patch_h}×{patch_w})', fontsize=14, fontweight='bold')
ax2.set_xlabel('패치 X 좌표')
ax2.set_ylabel('패치 Y 좌표')
plt.colorbar(im2, ax=ax2, fraction=0.046)

# 3. 보간 품질 곡선
ax3 = axes[1, 0]
ax3.plot(interp_df['interpolation_ratio'], interp_df['mse_loss'], 'o-', 
         linewidth=2, markersize=8, label='MSE Loss')
ax3.set_xlabel('보간 비율 (목표패치수/원본패치수)', fontsize=12)
ax3.set_ylabel('MSE Loss', fontsize=12)
ax3.set_title('보간 비율별 MSE 손실', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3)
ax3.legend()

# 해상도 라벨 추가
for _, row in interp_df.iterrows():
    ax3.annotate(row['resolution'], 
                (row['interpolation_ratio'], row['mse_loss']),
                xytext=(5, 5), textcoords='offset points', fontsize=9)

# 4. Cosine 유사도 곡선
ax4 = axes[1, 1]
ax4.plot(interp_df['interpolation_ratio'], interp_df['cosine_similarity'], 's-',
         color='orange', linewidth=2, markersize=8, label='Cosine Similarity')
ax4.set_xlabel('보간 비율', fontsize=12)
ax4.set_ylabel('Cosine 유사도', fontsize=12)
ax4.set_title('보간 비율별 Cosine 유사도', fontsize=14, fontweight='bold')
ax4.grid(True, alpha=0.3)
ax4.legend()

# 해상도 라벨 추가
for _, row in interp_df.iterrows():
    ax4.annotate(row['resolution'],
                (row['interpolation_ratio'], row['cosine_similarity']),
                xytext=(5, 5), textcoords='offset points', fontsize=9)

# 5. Sin-cos vs 학습된 임베딩 비교 (224×224)
ax5 = axes[2, 0]
learned_embed = analyzer.original_pos_embed[0, 1:]  # CLS 제외
sincos_embed = analyzer.create_2d_sincos_embedding(14, 14)[0, 1:]

# 첫 몇 개 차원의 분포 비교
learned_sample = learned_embed[:50, :8].cpu().numpy()  # 첫 50개 패치, 8개 차원
sincos_sample = sincos_embed[:50, :8].cpu().numpy()

x_pos = np.arange(len(learned_sample))
width = 0.35

bars1 = ax5.bar(x_pos - width/2, learned_sample.mean(axis=1), width, 
                label='학습된 임베딩', alpha=0.8)
bars2 = ax5.bar(x_pos + width/2, sincos_sample.mean(axis=1), width,
                label='Sin-cos 임베딩', alpha=0.8)

ax5.set_xlabel('패치 인덱스 (처음 50개)', fontsize=12)
ax5.set_ylabel('임베딩 평균값', fontsize=12)
ax5.set_title('학습된 vs Sin-cos 위치 임베딩 비교', fontsize=14, fontweight='bold')
ax5.legend()
ax5.grid(True, alpha=0.3)

# 6. L2 거리 변화
ax6 = axes[2, 1]
bars = ax6.bar(range(len(interp_df)), interp_df['l2_norm_diff'], 
               color='crimson', alpha=0.7)
ax6.set_xlabel('해상도', fontsize=12)
ax6.set_ylabel('L2 Norm 차이', fontsize=12)
ax6.set_title('해상도별 보간 L2 거리', fontsize=14, fontweight='bold')
ax6.set_xticks(range(len(interp_df)))
ax6.set_xticklabels(interp_df['resolution'], rotation=45)
ax6.grid(True, alpha=0.3)

# 막대 위에 값 표시
for i, (bar, val) in enumerate(zip(bars, interp_df['l2_norm_diff'])):
    ax6.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,
             f'{val:.2f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('../reports/figures/pe_heatmap_v2.png', dpi=150, bbox_inches='tight')
plt.show()

print("💾 위치 임베딩 히트맵 저장: pe_heatmap_v2.png")


In [ ]:
# 보간 품질 곡선 별도 저장
print("📈 보간 품질 곡선 생성 중...")

plt.figure(figsize=(14, 10))

# 2x2 서브플롯
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. MSE vs 보간 비율
ax1 = axes[0, 0]
ax1.plot(interp_df['interpolation_ratio'], interp_df['mse_loss'], 'o-',
         linewidth=3, markersize=10, color='steelblue')
ax1.set_xlabel('보간 비율 (목표/원본 패치수)', fontsize=12, fontweight='bold')
ax1.set_ylabel('MSE Loss', fontsize=12, fontweight='bold')
ax1.set_title('보간 비율별 MSE 손실', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

# 라벨 추가
for _, row in interp_df.iterrows():
    ax1.annotate(row['resolution'], 
                (row['interpolation_ratio'], row['mse_loss']),
                xytext=(8, 8), textcoords='offset points', 
                fontsize=11, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))

# 2. Cosine 유사도 vs 보간 비율
ax2 = axes[0, 1]
ax2.plot(interp_df['interpolation_ratio'], interp_df['cosine_similarity'], 's-',
         linewidth=3, markersize=10, color='orange')
ax2.set_xlabel('보간 비율', fontsize=12, fontweight='bold')
ax2.set_ylabel('Cosine 유사도', fontsize=12, fontweight='bold')
ax2.set_title('보간 비율별 Cosine 유사도', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

# 라벨 추가
for _, row in interp_df.iterrows():
    ax2.annotate(row['resolution'],
                (row['interpolation_ratio'], row['cosine_similarity']),
                xytext=(8, 8), textcoords='offset points',
                fontsize=11, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgreen', alpha=0.7))

# 3. L2 Norm vs 해상도
ax3 = axes[1, 0]
colors = plt.cm.viridis(np.linspace(0, 1, len(interp_df)))
bars = ax3.bar(range(len(interp_df)), interp_df['l2_norm_diff'], 
               color=colors, alpha=0.8, edgecolor='black', linewidth=1)
ax3.set_xlabel('해상도', fontsize=12, fontweight='bold')
ax3.set_ylabel('L2 Norm 차이', fontsize=12, fontweight='bold')
ax3.set_title('해상도별 보간 L2 거리', fontsize=14, fontweight='bold')
ax3.set_xticks(range(len(interp_df)))
ax3.set_xticklabels(interp_df['resolution'], rotation=45)
ax3.grid(True, alpha=0.3)

# 막대 위에 값 표시
for i, (bar, val) in enumerate(zip(bars, interp_df['l2_norm_diff'])):
    ax3.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
             f'{val:.1f}', ha='center', va='bottom', 
             fontsize=10, fontweight='bold')

# 4. 종합 품질 지표 (MSE vs Cosine)
ax4 = axes[1, 1]
scatter = ax4.scatter(interp_df['mse_loss'], interp_df['cosine_similarity'],
                     s=interp_df['target_patches']*3, # 패치 수에 비례하는 크기
                     c=interp_df['interpolation_ratio'], 
                     cmap='plasma', alpha=0.8, edgecolors='black')

ax4.set_xlabel('MSE Loss', fontsize=12, fontweight='bold')
ax4.set_ylabel('Cosine 유사도', fontsize=12, fontweight='bold')
ax4.set_title('보간 품질 종합 분석', fontsize=14, fontweight='bold')
ax4.grid(True, alpha=0.3)

# 컬러바 추가
cbar = plt.colorbar(scatter, ax=ax4)
cbar.set_label('보간 비율', fontsize=11)

# 라벨 추가
for _, row in interp_df.iterrows():
    ax4.annotate(row['resolution'],
                (row['mse_loss'], row['cosine_similarity']),
                xytext=(5, 5), textcoords='offset points',
                fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/pos_interp_curve_v2.png', dpi=150, bbox_inches='tight')
plt.show()

print("💾 보간 품질 곡선 저장: pos_interp_curve_v2.png")


## 📋 실험 결과 요약

**주요 산출물**:
1. **pos_embedding_analysis_v2.csv**: 해상도별 보간 품질 정량 분석
2. **pe_heatmap_v2.png**: 위치 임베딩 히트맵 및 종합 분석
3. **pos_interp_curve_v2.png**: 보간 품질 곡선 상세 분석

**핵심 인사이트**:

### 🔍 위치 임베딩 특성
- **학습된 패턴**: 원본 위치 임베딩은 공간적 지역성을 잘 보존
- **보간 안정성**: Bicubic 보간이 위치 정보를 효과적으로 유지
- **해상도 확장성**: 2배 이상 해상도에서도 합리적인 보간 품질

### 📊 보간 품질 분석
- **MSE 손실**: 보간 비율 증가 시 점진적 품질 저하
- **Cosine 유사도**: 대부분 해상도에서 0.95 이상 유지
- **L2 거리**: 극단적 해상도 변경에서만 큰 변화

### ⚙️ 실용적 가이드
- **안전 범위**: 원본 대비 4배 이하 패치 증가 권장
- **직사각형 처리**: 가로세로 비율 변경도 안정적 지원
- **실시간 적용**: 보간 오버헤드 최소화로 실용적 사용 가능

**권장 설정**:
- **일반 용도**: 512×512 이하에서 bicubic 보간 사용
- **품질 중심**: MSE < 0.01 기준으로 해상도 제한
- **성능 중심**: Cosine 유사도 > 0.95 기준 적용

**다음 단계**: CIFAR-100 파인튜닝 실험 (`05_train_cifar100_deit.ipynb`)
